# Bird Translation AI: Colab Runner

Run this pipeline in Colab while continuing to develop code locally.

Source modes:
- `upload_zip`: upload a local bundle generated by `colab/make_bundle.sh`
- `github`: clone from your GitHub repository


In [ ]:
CONFIG = {
    "source_mode": "upload_zip",  # "upload_zip" or "github"
    "github_repo": "https://github.com/<your-user>/<your-repo>.git",
    "github_ref": "main",
    "project_dir": "/content/bird_translation_ai",
    "drive_root": "/content/drive/MyDrive/BirdTranslationAI",
    "input_subdir": "raw",
    "run_name": "run_01",
}
CONFIG


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
import shutil
import subprocess
from pathlib import Path

project_dir = Path(CONFIG["project_dir"])
if project_dir.exists():
    shutil.rmtree(project_dir)
project_dir.mkdir(parents=True, exist_ok=True)

def run(cmd, cwd=None):
    print('+', ' '.join(cmd))
    subprocess.run(cmd, cwd=cwd, check=True)

if CONFIG["source_mode"] == "github":
    run([
        "git",
        "clone",
        "--depth",
        "1",
        "--branch",
        CONFIG["github_ref"],
        CONFIG["github_repo"],
        str(project_dir),
    ])
elif CONFIG["source_mode"] == "upload_zip":
    from google.colab import files

    uploaded = files.upload()
    if not uploaded:
        raise RuntimeError('No zip uploaded.')
    zip_name = next(iter(uploaded))
    run(["unzip", "-q", zip_name, "-d", str(project_dir)])
else:
    raise ValueError("CONFIG['source_mode'] must be 'github' or 'upload_zip'.")

print('Project ready at:', project_dir)
print('Top-level files:', sorted(p.name for p in project_dir.iterdir()))


In [ ]:
import shutil
import subprocess

def run(cmd):
    print('+', ' '.join(cmd))
    subprocess.run(cmd, check=True)

run(["apt-get", "update", "-y"])
run(["apt-get", "install", "-y", "ffmpeg"])
run(["python3", "-m", "pip", "install", "--upgrade", "pip"])
run(["python3", "-m", "pip", "install", "-r", str(project_dir / "requirements.txt")])
run(["python3", "-m", "pip", "install", "birdnet-analyzer[embeddings]", "perch-hoplite", "ml-collections"])

print('birdnet-analyze binary:', shutil.which('birdnet-analyze'))
print('birdnet-embeddings binary:', shutil.which('birdnet-embeddings'))


In [ ]:
drive_root = Path(CONFIG["drive_root"])
input_dir = drive_root / CONFIG["input_subdir"]
output_dir = drive_root / "runs" / CONFIG["run_name"]

input_dir.mkdir(parents=True, exist_ok=True)
output_dir.mkdir(parents=True, exist_ok=True)

print('Input directory:', input_dir)
print('Output directory:', output_dir)
print('Add source audio files to input directory before running the pipeline.')


In [ ]:
import subprocess

cmd = [
    "python3",
    "-m",
    "src.pipeline",
    "--input_dir",
    str(input_dir),
    "--output_dir",
    str(output_dir),
    "--sr",
    "32000",
    "--min_dur",
    "0.25",
    "--max_dur",
    "4.0",
    "--seg_threshold_db",
    "8",
    "--merge_gap",
    "0.25",
    "--birdiness_min_hz",
    "700",
    "--birdiness_ratio_min",
    "0.0",
    "--seg_rms_min",
    "0.003",
    "--conf_thresh",
    "0.25",
    "--top_k",
    "5",
    "--birdnet_binary",
    "birdnet-analyze",
    "--birdnet_embeddings_binary",
    "birdnet-embeddings",
    "--min_species_samples",
    "5",
    "--hdbscan_min_cluster_size",
    "0",
    "--hdbscan_min_samples",
    "0",
    "--embedding_backend",
    "birdnet",
    "--embedding_fallback_backend",
    "mel_time",
    "--embedding_mel_frames",
    "96",
]

print('+', ' '.join(cmd))
subprocess.run(cmd, cwd=project_dir, check=True)


In [ ]:
report_path = output_dir / "reports" / "index.html"
print('Report:', report_path)
print('Predictions CSV:', output_dir / "preds" / "master_predictions.csv")
print('Clusters CSV:', output_dir / "clusters" / "master_clusters.csv")
print('Clustered features CSV:', output_dir / "features" / "master_features_clustered.csv")


In [ ]:
# Optional: zip and download a full run output
import shutil
from google.colab import files

archive_path = shutil.make_archive('/content/bird_translation_outputs', 'zip', root_dir=output_dir)
print('Created archive:', archive_path)
files.download(archive_path)
